In [1]:
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import balanced_accuracy_score
import warnings

warnings.filterwarnings('ignore')

file_path = '../../../data/data_for_final_models/AgglomerativeClustering_generated_features.csv'
data = pd.read_csv(file_path)

X = data.drop(columns='Cluster')
y = data['Cluster']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

catboost_model = CatBoostClassifier(random_state=42, silent=True)
lr_model = LogisticRegression(random_state=42)
rf_model = RandomForestClassifier(random_state=42)
et_model = ExtraTreesClassifier(random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)

bagging_lr = BaggingClassifier(lr_model, random_state=42)
bagging_rf = BaggingClassifier(rf_model, random_state=42)
bagging_et = BaggingClassifier(et_model, random_state=42)
bagging_gb = BaggingClassifier(gb_model, random_state=42)

meta_preds = []
meta_labels = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Обучение базовых моделей
    catboost_model.fit(X_train, y_train)
    bagging_lr.fit(X_train, y_train)
    bagging_rf.fit(X_train, y_train)
    bagging_et.fit(X_train, y_train)
    bagging_gb.fit(X_train, y_train)
    
    # Получение предсказаний для мета-признаков
    catboost_pred = catboost_model.predict(X_val)
    lr_pred = bagging_lr.predict(X_val)
    rf_pred = bagging_rf.predict(X_val)
    et_pred = bagging_et.predict(X_val)
    gb_pred = bagging_gb.predict(X_val)
    
    # Сбор мета-признаков
    X_meta_fold = pd.DataFrame({
        'catboost': catboost_pred,
        'lr': lr_pred,
        'rf': rf_pred,
        'et': et_pred,
        'gb': gb_pred
    })
    X_meta_fold['catboost'] = X_meta_fold['catboost'].apply(lambda x: x[0])
    
    # Обучение мета-модели на текущих предсказаниях
    meta_model = LogisticRegression(random_state=42)
    meta_model.fit(X_meta_fold, y_val)
    
    # Предсказание мета-модели и сохранение результатов
    fold_preds = meta_model.predict(X_meta_fold)
    meta_preds.extend(fold_preds)
    meta_labels.extend(y_val)

# Вычисление итоговой сбалансированной точности
final_accuracy = balanced_accuracy_score(meta_labels, meta_preds)
print(f'Mean Balanced Accuracy: {final_accuracy:.4f}')